In [ ]:
import boto3
from dotenv import load_dotenv
import os
import io
from PIL import Image
from tqdm import tqdm
from IPython.display import clear_output

load_dotenv("../../backend/.env")

session = boto3.Session(
    aws_access_key_id=os.getenv('ACCESS_KEY'),
    aws_secret_access_key=os.getenv('SECRET_ACCESS_KEY'),
    aws_session_token=os.getenv('SESSION_TOKEN')
)

s3 = session.resource('s3')
path = "data/token/=/"
bucket = s3.Bucket("penman-lln")

objs = bucket.objects.filter(Prefix=path)

In [ ]:
# Removes unwanted images from the existing S3 bucket.

for i, obj in tqdm(enumerate(objs)):
    if obj.key == path:
        continue

    curr_url = s3.Object("penman-lln", obj.key).get()["Body"].read()
    image = Image.open(io.BytesIO(curr_url))

    display(image)

    confirm = bool(input("Remove image?"))

    if confirm:
        s3.Object('penman-lln', obj.key).delete()

    clear_output(wait=True)

In [ ]:
# Renames all items.

for i, obj in tqdm(enumerate(objs)):
    if obj.key == path:
        continue

    curr_url = s3.Object("penman-lln", obj.key).get()["Body"].read()
    
    new_key = obj.key.replace("equation", "token")

    s3.Object('penman-lln', new_key).put(
        Body=curr_url,
        ContentType="image/png"
    )

    s3.Object('penman-lln', obj.key).delete()